In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from spread_backtester import (
    StrategyConfig, SpreadBacktester, load_price_data,
    train_test_split_test, walk_forward_test,
)

pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
data = load_price_data(
    mt5_csv_path="USDJPYM1.csv",   
    hl_coin="xyz:JPY",
    hl_interval="1m",
    mt5_tz=None,   
)

print(f"{len(data):,} aligned bars")
print(f"{data.index.min()}  ->  {data.index.max()}")
data.tail()


In [ ]:
# NEW: Momentum Configuration with Trailing Stops and Scale-Outs
cfg = StrategyConfig(
    entry_sigma=1.5,                 # Enter when spread breaks 1.5 SDs
    tp_levels_sigma=[1.0, 2.0, 3.0], # TP at +1.0, +2.0, and +3.0 SDs from entry
    tp_allocations=[0.33, 0.33, 0.34],# Close 33% at TP1, 33% at TP2, 34% at TP3
    trailing_stop_sigma=0.5,         # Trail stop 0.5 SDs behind the peak spread
    risk_pct_per_trade=0.01,
    max_drawdown_pct=0.15,           
    order_type="market",
    rolling=True,                   
    lookback_bars=720,               # Halved lookback (12 hrs) to react faster to breakouts
    starting_equity=10_000.0,
)
cfg





In [ ]:
bt_full = SpreadBacktester(data, cfg)
summary_full = bt_full.run()

for k, v in summary_full.items():
    print(f"{k:>22}: {v}")

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=False)

d = bt_full.bands_df
axes[0].plot(d.index, d["spread"], color="orange", linewidth=0.6, label="spread")
axes[0].plot(d.index, d["mu"], color="black", linewidth=0.8, label="mu")
axes[0].plot(d.index, d["buy_entry"], "--", color="green", linewidth=0.6, label="long breakout")
axes[0].plot(d.index, d["sell_entry"], "--", color="red", linewidth=0.6, label="short breakout")

for t in bt_full.trades:
    marker = "^" if t.side == "long" else "v"
    color = "green" if t.side == "long" else "red"
    axes[0].scatter(t.entry_time, t.entry_price, marker=marker, color=color, s=35, zorder=5)
    if t.exit_time:
        # Mark exits (including partials if your backtester logs them)
        axes[0].scatter(t.exit_time, t.exit_price, marker="x", color="black", s=25, zorder=5)
        
axes[0].set_title("Spread Momentum Breakout — Full Period")
axes[0].legend(fontsize=8, loc="upper right")

axes[1].plot(bt_full.equity_df.index, bt_full.equity_df["equity"], color="blue")
axes[1].set_title("Equity Curve — Trailing Stop & Scale-Out")
if bt_full.halted_at:
    axes[1].axvline(bt_full.halted_at, color="red", linestyle="--", label="kill switch")
    axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
oos = train_test_split_test(data, cfg, train_frac=0.5)
bt_train, stats_train = oos["train"]
bt_test, stats_test = oos["test"]

comparison = pd.DataFrame([stats_train, stats_test], index=["train (in-sample)", "test (out-of-sample)"])
display(comparison[["trades", "win_rate_pct", "total_net_pnl", "max_drawdown_pct", "sharpe", "halted_by_kill_switch"]])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=False)
axes[0].plot(bt_train.equity_df.index, bt_train.equity_df["equity"], color="steelblue")
axes[0].set_title("Train (in-sample) equity curve")
axes[1].plot(bt_test.equity_df.index, bt_test.equity_df["equity"], color="darkorange")
axes[1].set_title("Test (out-of-sample) equity curve")
plt.tight_layout()
plt.show()

In [ ]:
# n_folds set to 3 to ensure lookback_bars (720) fits safely inside each fold
wf = walk_forward_test(data, cfg, n_folds=3, compound=True)
fold_stats = wf["fold_stats"]

cols = ["fold", "fold_start", "fold_end", "trades", "win_rate_pct",
        "total_net_pnl", "max_drawdown_pct", "sharpe", "ending_equity"]
display(fold_stats[cols])

fig, ax = plt.subplots(figsize=(13, 5))

offset = 0
boundaries = []
for bt in wf["backtesters"]:
    eq = bt.equity_df["equity"]
    x = range(offset, offset + len(eq))
    ax.plot(x, eq.values, linewidth=1)
    offset += len(eq)
    boundaries.append(offset)

for b in boundaries[:-1]:
    ax.axvline(b, color="gray", linestyle=":", linewidth=0.8)

ax.set_title("Walk-Forward Equity Curve (Momentum Breakout)")
ax.set_xlabel("Bar index")
ax.set_ylabel("Equity")
plt.tight_layout()
plt.show()

print(f"Win rate across folds: mean {fold_stats['win_rate_pct'].mean():.1f}%, "
      f"std {fold_stats['win_rate_pct'].std():.1f}pp")
print(f"Net PnL across folds:  mean {fold_stats['total_net_pnl'].mean():.2f}, "
      f"std {fold_stats['total_net_pnl'].std():.2f}")